# Collaborative Filtering Experiments

Notebook này triển khai và đánh giá 2 phương pháp Collaborative Filtering:
1. **Item-based CF**: Cosine similarity giữa item vectors
2. **ALS (Alternating Least Squares)**: Matrix factorization cho implicit feedback

## Nội dung
1. Setup & Import
2. Load Data
3. Item-based CF
4. ALS (baseline)
5. ALS Tuning (Grid Search)
6. So sánh với Popularity Baseline
7. Visualization
8. Case Study
9. Export Best Models

## 1. Setup & Import

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Add project root to path
_current = Path.cwd().resolve()
PROJECT_ROOT = _current
while PROJECT_ROOT.name != "RecommendationSystem" and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import modules
from src.data_loader import ML25MLoader, DataPreprocessor
from src.recommenders.item_cf import ItemCFRecommender
from src.recommenders.als_recommender import ALSRecommender
from src.evaluation import EvaluationPipeline
from src.recommenders import (
    PopularityRecommender,
    BayesianPopularityRecommender,
    WilsonLowerBoundRecommender
)

# Set visualization style 
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Import thành công tất cả thư viện")

✓ Import thành công tất cả thư viện


/home/huyquang/Workspace/Personal/PyCharmProjects/RecommendationSystem/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load Data

Load processed train/test splits từ `data/processed/`

In [2]:
# Load processed splits
train_df = pd.read_csv(PROJECT_ROOT / 'data/processed/train_ml25m.csv')
test_df = pd.read_csv(PROJECT_ROOT / 'data/processed/test_ml25m.csv')

# Load movies metadata for case study
movies_df = pd.read_csv(PROJECT_ROOT / 'data/raw/ml-25m/movies.csv')

# Statistics
print(f"Train: {len(train_df):,} ratings")
print(f"Test: {len(test_df):,} ratings")
print(f"Users: {train_df['userId'].nunique():,}")
print(f"Movies: {train_df['movieId'].nunique():,}")
print(f"Sparsity: {100 * len(train_df) / (train_df['userId'].nunique() * train_df['movieId'].nunique()):.4f}%")

Train: 17,500,066 ratings
Test: 548,359 ratings
Users: 122,029
Movies: 22,931
Sparsity: 0.6254%


## 3. Item-based Collaborative Filtering

Item-based CF tính similarity giữa items dựa trên user rating patterns

In [3]:
# Build explicit user-item matrix
preprocessor = DataPreprocessor()
user_item_csr, user_map, movie_map = preprocessor.build_user_item_matrix(
    train_df, value_column='rating'
)

# Train ItemCF
print("\nTraining Item-based CF...")
item_cf = ItemCFRecommender()
item_cf.fit(user_item_csr, user_map, movie_map)

# Evaluate
print("\nEvaluating Item-based CF...")
evaluator = EvaluationPipeline(train_df, test_df, positive_threshold=4.0)
item_cf_results = evaluator.evaluate_model(item_cf, model_name="ItemCF")

User-item matrix: (122029, 22931), sparsity: 0.6254%, nnz: 17,500,066

Training Item-based CF...
ItemCF fitted: 22931 items, 122029 users

Evaluating Item-based CF...
Ground truth prepared: 4,485 users with relevant items
Avg relevant items per user: 47.30
Seen items prepared: 122,029 users
Avg seen items per user: 143.4
  Evaluated 1,000/4,485 users...
  Evaluated 2,000/4,485 users...
  Evaluated 3,000/4,485 users...
  Evaluated 4,000/4,485 users...

ItemCF Evaluation Complete:
  Users evaluated: 4,485
  Users with no recommendations: 0
  Precision@5: 0.1648
  Recall@5: 0.0245
  NDCG@5: 0.1764
  Precision@10: 0.1441
  Recall@10: 0.0412
  NDCG@10: 0.1612
  Precision@20: 0.1228
  Recall@20: 0.0676
  NDCG@20: 0.1488


## 4. ALS (Alternating Least Squares)

ALS matrix factorization cho implicit feedback (ratings >= 4.0)

In [4]:
# Convert to implicit feedback
train_implicit = preprocessor.to_implicit(train_df, threshold=4.0)

# Train ALS with baseline params
print("\nTraining ALS (baseline)...")
als_baseline = ALSRecommender(factors=64, regularization=0.01, iterations=15, alpha=1.0)
als_baseline.fit(train_implicit)

# Evaluate
print("\nEvaluating ALS (baseline)...")
als_baseline_results = evaluator.evaluate_model(als_baseline, model_name="ALS-baseline")

Implicit conversion: 8,750,986 / 17,500,066 (50.0%) positive interactions

Training ALS (baseline)...


/home/huyquang/Workspace/Personal/PyCharmProjects/RecommendationSystem/venv/lib/python3.12/site-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 12 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


User-item matrix: (121829, 18405), sparsity: 0.3903%, nnz: 8,750,986
Training ALS: factors=64, reg=0.01, iters=15, alpha=1.0


100%|██████████| 15/15 [00:14<00:00,  1.06it/s, loss=0.00784]


ALS training complete

Evaluating ALS (baseline)...
  Evaluated 1,000/4,485 users...
  Evaluated 2,000/4,485 users...
  Evaluated 3,000/4,485 users...
  Evaluated 4,000/4,485 users...

ALS-baseline Evaluation Complete:
  Users evaluated: 4,485
  Users with no recommendations: 59
  Precision@5: 0.2157
  Recall@5: 0.0339
  NDCG@5: 0.2240
  Precision@10: 0.1953
  Recall@10: 0.0548
  NDCG@10: 0.2002
  Precision@20: 0.1823
  Recall@20: 0.0712
  NDCG@20: 0.1637


## 5. ALS Hyperparameter Tuning

Grid search trên validation set (80/20 split từ train data)

In [ ]:
# Split train into train_sub + validation theo TEMPORAL split.
# Van de cu: per-user random split co the tao temporal leakage - val rating cua user X
# co timestamp cu hon train_sub rating cung user X, khien metric val over-optimistic.
# Giai phap: dung DataPreprocessor.temporal_user_split - sort theo timestamp trong
# tung user, lay 20% rating MOI NHAT lam val. Val luon nam "trong tuong lai" cua train_sub.
# Singleton user (1 positive) giu nguyen trong train_sub (giong logic cu).
train_sub_df, val_df, _singleton_users = DataPreprocessor.temporal_user_split(
    train_implicit, val_ratio=0.2
)

if _singleton_users:
    print(f"{_singleton_users:,} users have only 1 positive "
          f"interaction -> kept entirely in train_sub (no held-out).")

print(f"Train subset: {len(train_sub_df):,} ratings")
print(f"Validation:   {len(val_df):,} ratings "
      f"({len(val_df)/(len(train_sub_df)+len(val_df))*100:.2f}%)")

# Filter val_df: keep only warm users/items present in train_sub
train_users = set(train_sub_df['userId'].unique())
train_movies = set(train_sub_df['movieId'].unique())
val_df = val_df[
    val_df['userId'].isin(train_users) & val_df['movieId'].isin(train_movies)
].reset_index(drop=True)
print(f"Validation (warm-only): {len(val_df):,} ratings")

# Create validation evaluator
val_evaluator = EvaluationPipeline(train_sub_df, val_df, positive_threshold=4.0)


In [ ]:
# Grid search parameters - MỞ RỘNG (180 combinations)
param_grid = {
    'factors': [16, 32, 64, 128, 256],
    'regularization': [0.001, 0.01, 0.05, 0.1],
    'iterations': [15, 25, 40],
    'alpha': [1, 15, 40]  # Thêm mới - confidence scaling
}

from joblib import Parallel, delayed

def train_and_eval(factors, reg, iters, alpha, train_sub_df, val_evaluator):
    als = ALSRecommender(factors=factors, regularization=reg, iterations=iters, alpha=alpha)
    als.fit(train_sub_df)
    result = val_evaluator.evaluate_model(als, model_name=f"ALS-{factors}-{reg}-{iters}-{alpha}")
    # result['metrics'] chứa các metric values
    return {
        'factors': factors,
        'regularization': reg,
        'iterations': iters,
        'alpha': alpha,
        'NDCG@10': result['metrics']['NDCG@10'],
        'Recall@10': result['metrics']['Recall@10'],
        'Precision@10': result['metrics']['Precision@10']
    }

print(f"\nGrid Search trên Validation Set ({5*4*3*3} combinations)...")
results = Parallel(n_jobs=-1, verbose=10)(
    delayed(train_and_eval)(f, r, i, a, train_sub_df, val_evaluator)
    for f in param_grid['factors']
    for r in param_grid['regularization']
    for i in param_grid['iterations']
    for a in param_grid['alpha']
)

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('NDCG@10', ascending=False)
print("\n=== Top 10 Configurations (by NDCG@10) ===")
print(results_df.head(10).to_string(index=False))


### 5.1 Final Test trên `test_ml25m.csv`

Sau khi chốt best params từ validation, ta **retrain trên toàn bộ** `train_implicit`
và đánh giá trên `test_ml25m.csv` (đã có sẵn từ notebook 00 - 30% tương lai + filter cold-start).

**Đây là final test, dùng đúng 1 lần** để báo cáo metric cuối cùng:
- `test_ml25m.csv` đã được tách theo **temporal split** ở notebook 00 (30% rating mới nhất)
- Đã **filter cold-start** ở notebook 00 -> chỉ chứa warm user/item
- KHÔNG dùng `test_ml25m.csv` cho hyperparameter tuning (đã dùng `val_df` ở trên)

Metric ở đây phản ánh khả năng dự đoán tương lai của model trong production-like setting.

In [ ]:
# Get best parameters
best_params = results_df.iloc[0]
print(f"\n=== Best Parameters ===")
print(f"Factors: {int(best_params['factors'])}")
print(f"Regularization: {best_params['regularization']}")
print(f"Iterations: {int(best_params['iterations'])}")
print(f"Alpha: {best_params['alpha']}")
print(f"Val NDCG@10: {best_params['NDCG@10']:.4f}")

# Retrain with best params on full train data
print(f"\nRetraining với best params trên full train data...")
als_best = ALSRecommender(
    factors=int(best_params['factors']),
    regularization=best_params['regularization'],
    iterations=int(best_params['iterations']),
    alpha=best_params['alpha']
)
als_best.fit(train_implicit)

# Evaluate on test set
print("\nEvaluating trên test set...")
als_best_results = evaluator.evaluate_model(als_best, model_name="ALS-best")


## 6. So sánh với Popularity Baseline

Load popularity baseline từ notebook 01 để so sánh

In [ ]:
# Train popularity baselines for comparison
print("Training Popularity Baselines...")

pop_global = PopularityRecommender()
pop_global.fit(train_df)
pop_global_results = evaluator.evaluate_model(pop_global, model_name="Popularity-Global")

pop_weighted = BayesianPopularityRecommender()
pop_weighted.fit(train_df)
pop_weighted_results = evaluator.evaluate_model(pop_weighted, model_name="Popularity-Weighted")

# Compile all results
all_results = pd.DataFrame([
    pop_global_results,
    pop_weighted_results,
    item_cf_results,
    als_baseline_results,
    als_best_results
])

print("\n=== So sánh tất cả Models ===")
print(all_results.to_string(index=False))

## 7. Visualization

Visualize so sánh performance giữa các models

In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics = ['NDCG@10', 'Recall@10', 'Precision@10']
for i, metric in enumerate(metrics):
    ax = axes[i]
    all_results.plot(x='Model', y=metric, kind='bar', ax=ax, color='steelblue', legend=False)
    ax.set_title(f'{metric} Comparison', fontsize=14, fontweight='bold')
    ax.set_ylabel(metric)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'reports/figures/cf_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Đã lưu biểu đồ vào reports/figures/cf_comparison.png")

In [ ]:
# Heatmap for grid search results
pivot_table = results_df.pivot_table(
    values='NDCG@10',
    index=['factors', 'regularization'],
    columns='iterations',
    aggfunc='mean'
)

plt.figure(figsize=(10, 8))
sns.heatmap(pivot_table, annot=True, fmt='.4f', cmap='YlOrRd', cbar_kws={'label': 'NDCG@10'})
plt.title('ALS Grid Search Results (NDCG@10)', fontsize=14, fontweight='bold')
plt.xlabel('Iterations')
plt.ylabel('(Factors, Regularization)')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'reports/figures/als_grid_search_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Đã lưu heatmap vào reports/figures/als_grid_search_heatmap.png")

## 8. Case Study

Kiểm tra recommendations cho một vài sample users

In [ ]:
# Helper function to show recommendations
def show_recommendations(user_id, models_dict, n=10):
    """
    Show recommendations from multiple models for a single user
    
    Args:
        user_id: User ID
        models_dict: Dict of {model_name: model_object}
        n: Number of recommendations
    """
    print(f"\n{'='*80}")
    print(f"User {user_id} - Recommendations Comparison")
    print(f"{'='*80}")
    
    # Show user's historical ratings
    user_history = train_df[train_df['userId'] == user_id].sort_values('rating', ascending=False).head(5)
    print("\nUser's Top 5 Rated Movies (History):")
    for _, row in user_history.iterrows():
        movie_title = movies_df[movies_df['movieId'] == row['movieId']]['title'].values
        if len(movie_title) > 0:
            print(f"  • {movie_title[0]} (Rating: {row['rating']:.1f})")
    
    print("\n" + "-"*80)
    
    # Get user's seen items
    seen = set(train_df[train_df['userId'] == user_id]['movieId'].values)
    
    # Get recommendations from each model
    for model_name, model in models_dict.items():
        print(f"\n{model_name}:")
        try:
            # Models return list of movieIds
            recs = model.recommend(user_id, n=n, seen=seen)
            
            if not recs:
                print("  No recommendations available")
                continue
                
            for i, movie_id in enumerate(recs, 1):
                movie_title = movies_df[movies_df['movieId'] == movie_id]['title'].values
                if len(movie_title) > 0:
                    print(f"  {i}. {movie_title[0]}")
                else:
                    print(f"  {i}. Movie ID {movie_id}")
        except Exception as e:
            print(f"  Error: {e}")

# Sample users for case study
sample_users = train_df['userId'].sample(3, random_state=42).tolist()

models_dict = {
    'Popularity-Global': pop_global,
    'Item-CF': item_cf,
    'ALS-best': als_best
}

for user_id in sample_users:
    show_recommendations(user_id, models_dict, n=10)

## 9. Export Best Models

Lưu best models để sử dụng sau này (deployment, inference)

In [ ]:
# Create models directory if not exists
models_dir = PROJECT_ROOT / 'models'
models_dir.mkdir(exist_ok=True)

# Save best models
print("Exporting models...")

# 1. Item-CF
item_cf.save(str(models_dir / 'item_cf_best.pkl'))
print(f"✓ Saved: {models_dir / 'item_cf_best.pkl'}")

# 2. ALS-best
als_best.save(str(models_dir / 'als_best.pkl'))
print(f"✓ Saved: {models_dir / 'als_best.pkl'}")

# 3. Popularity baseline (for comparison)
pop_weighted.save(str(models_dir / 'popularity_weighted.pkl'))
print(f"✓ Saved: {models_dir / 'popularity_weighted.pkl'}")

print("\nĐể load models sau này:")
print("  from src.recommenders import ItemCFRecommender, ALSRecommender")
print("  item_cf = ItemCFRecommender.load('models/item_cf_best.pkl')")
print("  als = ALSRecommender.load('models/als_best.pkl')")